In [21]:
import os
import sys
import traceback
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Definimos la configuración de iSpec
ISPEC_DIR = "/Users/carlasequero/iSpec"

if ISPEC_DIR not in sys.path:
    sys.path.insert(0, ISPEC_DIR)

import ispec

ISPEC_CONFIG = {

    "code": "spectrum",

    # Resolución aproximada de SDSS
    "resolution": 2000,

    # Rango de las muestras del conjunto de datos
    "wave_min_nm": 500.0,
    "wave_max_nm": 900.0,

    # Parámetros iniciales por defecto si no tenemos los de Gaia
    "default_teff": 5500.0,
    "default_logg": 4.0,
    "default_mh": 0.0,

 
    # Recursos de iSpec
    "atmosphere_dir":
        "/Users/carlasequero/iSpec/input/atmospheres/MARCS.GES/",

    "atomic_linelist_file":
        "/Users/carlasequero/iSpec/input/linelists/transitions/"
        "VALD.300_1100nm/atomic_lines.tsv",

    "solar_abundances_file":
        "/Users/carlasequero/iSpec/input/abundances/"
        "Grevesse.2007/stdatom.dat",

    "isotopes_file":
        "/Users/carlasequero/iSpec/input/isotopes/SPECTRUM.lst",

    "line_regions_file":
        "/Users/carlasequero/iSpec/input/regions/"
        "47000_SPECTRUM/"
        "spectrum_synth_good_for_params_all.txt",


    # Elimina de la lista atómica las líneas teóricamente demasiado débiles.
    "minimum_theoretical_depth": 0.01,

    # Número máximo de iteraciones del ajuste de iSpec
    "max_iterations": 6,
 
}

✓ Lista atómica: /Users/carlasequero/iSpec/input/linelists/transitions/VALD.300_1100nm/atomic_lines.tsv
✓ Abundancias solares: /Users/carlasequero/iSpec/input/abundances/Grevesse.2007/stdatom.dat
✓ Isótopos: /Users/carlasequero/iSpec/input/isotopes/SPECTRUM.lst
✓ Atmósferas: /Users/carlasequero/iSpec/input/atmospheres/MARCS.GES/
✓ Regiones: /Users/carlasequero/iSpec/input/regions/47000_SPECTRUM/spectrum_synth_good_for_params_all.txt


True

In [22]:
def load_ispec_resources(config=ISPEC_CONFIG):

    print("Cargando modelos atmosféricos...")
    modeled_layers_pack = ispec.load_modeled_layers_pack(
        config["atmosphere_dir"]
    )

    print("Cargando abundancias solares...")
    solar_abundances = ispec.read_solar_abundances(
        config["solar_abundances_file"]
    )

    print("Cargando isótopos...")
    isotopes = ispec.read_isotope_data(
        config["isotopes_file"]
    )

    print("Cargando lista atómica...")
    atomic_linelist = ispec.read_atomic_linelist(
        config["atomic_linelist_file"],
        wave_base=config["wave_min_nm"],
        wave_top=config["wave_max_nm"]
    )

    # Filtrar líneas extremadamente débiles
    if "theoretical_depth" in atomic_linelist.dtype.names:
        atomic_linelist = atomic_linelist[
            atomic_linelist["theoretical_depth"]
            >= config["minimum_theoretical_depth"]
        ]

    print(f"Líneas atómicas cargadas: {len(atomic_linelist)}")

    print("Cargando regiones espectrales...")
    line_regions = ispec.read_line_regions(
        config["line_regions_file"]
    )

    print(f"Regiones cargadas: {len(line_regions)}")

    print("\nRecursos de iSpec cargados correctamente.")

    return {
        "modeled_layers_pack": modeled_layers_pack,
        "solar_abundances": solar_abundances,
        "isotopes": isotopes,
        "atomic_linelist": atomic_linelist,
        "line_regions": line_regions
    }

In [23]:
resources = load_ispec_resources()

Cargando modelos atmosféricos...
Cargando abundancias solares...
Cargando isótopos...
Cargando lista atómica...
Líneas atómicas cargadas: 13195
Cargando regiones espectrales...
Regiones cargadas: 246

Recursos de iSpec cargados correctamente.


In [24]:
print(resources.keys())

dict_keys(['modeled_layers_pack', 'solar_abundances', 'isotopes', 'atomic_linelist', 'line_regions'])


In [ ]:
def build_ispec_spectrum(
    wavelength_aa,
    flux,
    error=None,
    wave_min_nm=500.0,
    wave_max_nm=900.0
):

    wavelength_aa = np.asarray(wavelength_aa, dtype=float)
    flux = np.asarray(flux, dtype=float)

    # Å -> nm
    wavelength_nm = wavelength_aa / 10.0

    if error is None:
        error = np.zeros_like(flux, dtype=float)
    else:
        error = np.asarray(error, dtype=float)

    # Filtrar valores inválidos
    valid = (
        np.isfinite(wavelength_nm)
        & np.isfinite(flux)
        & np.isfinite(error)
        & (wavelength_nm >= wave_min_nm)
        & (wavelength_nm <= wave_max_nm)
    )

    wavelength_nm = wavelength_nm[valid]
    flux = flux[valid]
    error = error[valid]

    if len(wavelength_nm) == 0:
        raise ValueError("El espectro no contiene puntos válidos.")

    # Ordenar por longitud de onda
    order = np.argsort(wavelength_nm)

    wavelength_nm = wavelength_nm[order]
    flux = flux[order]
    error = error[order]

    spectrum = np.recarray(
        len(wavelength_nm),
        dtype=[
            ("waveobs", float),
            ("flux", float),
            ("err", float)
        ]
    )

    spectrum["waveobs"] = wavelength_nm
    spectrum["flux"] = flux
    spectrum["err"] = error

    return spectrum